### NBA_API

In [1]:
from nba_api.stats.endpoints import LeagueGameFinder, BoxScoreTraditionalV2
import pandas as pd
import time
from datetime import datetime

def get_current_season_boxscores():
    # Get current season string based on month (NBA season starts in October)
    current_date = datetime.now()
    current_year = current_date.year
    season_start_year = current_year if current_date.month >= 10 else current_year - 1
    season = f"{season_start_year}-{str(season_start_year + 1)[-2:]}"
    
    # Get all games for the current season
    games = LeagueGameFinder(
        season_nullable=season,
        league_id_nullable="00",  # NBA games only
        date_to_nullable=current_date.strftime('%m/%d/%Y')  # Only games up to today
    ).get_data_frames()[0]
    
    # Sort games by date
    games = games.sort_values('GAME_DATE')
    
    # Initialize lists to store our data
    all_player_stats = []
    all_team_stats = []
    
    print(f"Found {len(games)} games played so far this season. Starting to fetch boxscores...")
    
    for idx, game in games.iterrows():
        game_id = game['GAME_ID']
        try:
            # Fetch boxscore data
            boxscore = BoxScoreTraditionalV2(game_id=game_id)
            
            # Get player stats
            player_stats = boxscore.player_stats.get_data_frame()
            player_stats['GAME_ID'] = game_id
            player_stats['GAME_DATE'] = game['GAME_DATE']
            all_player_stats.append(player_stats)
            
            # Get team stats
            team_stats = boxscore.team_stats.get_data_frame()
            team_stats['GAME_ID'] = game_id
            team_stats['GAME_DATE'] = game['GAME_DATE']
            all_team_stats.append(team_stats)
            
            # Add a small delay to avoid hitting rate limits
            time.sleep(0.6)
            
            if idx % 10 == 0:
                print(f"Processed {idx + 1} games...")
                
        except Exception as e:
            print(f"Error processing game {game_id}: {str(e)}")
            continue
    
    # Combine all data
    player_stats_df = pd.concat(all_player_stats, ignore_index=True)
    team_stats_df = pd.concat(all_team_stats, ignore_index=True)
    
    print(f"\nCompleted! Fetched data for {len(games)} games from {games['GAME_DATE'].min()} to {games['GAME_DATE'].max()}")
    return player_stats_df, team_stats_df

def save_to_csv(player_stats, team_stats, date_str=None):
    if date_str is None:
        date_str = datetime.now().strftime('%Y%m%d')
        
    player_stats.to_csv(f'nba_player_stats_{date_str}.csv', index=False)
    team_stats.to_csv(f'nba_team_stats_{date_str}.csv', index=False)
    print(f"Data saved to CSV files with prefix 'nba_player_stats_{date_str}' and 'nba_team_stats_{date_str}'")
    
# Usage example
if __name__ == "__main__":
    player_stats, team_stats = get_current_season_boxscores()
    save_to_csv(player_stats, team_stats)

Found 622 games played so far this season. Starting to fetch boxscores...
Processed 621 games...
Processed 611 games...
Processed 601 games...
Processed 591 games...
Processed 581 games...
Processed 571 games...
Processed 561 games...
Processed 551 games...
Processed 541 games...
Processed 531 games...
Processed 521 games...
Processed 511 games...
Processed 491 games...
Processed 501 games...
Processed 481 games...
Processed 461 games...
Processed 471 games...
Processed 451 games...
Processed 431 games...
Processed 441 games...
Processed 411 games...
Processed 421 games...
Processed 401 games...
Processed 381 games...
Processed 391 games...
Processed 371 games...
Processed 351 games...
Processed 361 games...
Processed 341 games...
Processed 321 games...
Processed 331 games...
Processed 311 games...
Processed 301 games...
Processed 291 games...
Processed 271 games...
Processed 261 games...
Processed 281 games...
Processed 241 games...
Processed 251 games...
Processed 231 games...
Proces

C:\Users\User\AppData\Local\Temp\ipykernel_26160\1706315258.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  player_stats_df = pd.concat(all_player_stats, ignore_index=True)
C:\Users\User\AppData\Local\Temp\ipykernel_26160\1706315258.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  team_stats_df = pd.concat(all_team_stats, ignore_index=True)



Completed! Fetched data for 622 games from 2024-10-04 to 2024-11-23
Data saved to CSV files with prefix 'nba_player_stats_20241123' and 'nba_team_stats_20241123'


### Script that fetches injury reports using the NBA API. 
The NBA API doesn't have a direct injury endpoint, but we can get this information through the TeamDashboardPlayerLast5Games endpoint which includes player availability status.

#### Code below does not capture the first team data

In [ ]:
import requests
from bs4 import BeautifulSoup
import pdfplumber
import csv
import os
import re

class NBAInjuryReportScraper:
    nba_teams = [
        "Brooklyn Nets", "Philadelphia 76ers", "Boston Celtics", "Washington Wizards",
        "Golden State Warriors", "New Orleans Pelicans", "Atlanta Hawks", "Chicago Bulls",
        "Indiana Pacers", "Milwaukee Bucks", "Portland Trail Blazers", "Houston Rockets",
        "Dallas Mavericks", "Denver Nuggets", "Sacramento Kings", "LA Clippers",
        "New York Knicks", "Utah Jazz", "Detroit Pistons", "Orlando Magic", "Charlotte Hornets",
        "Memphis Grizzlies", "San Antonio Spurs", "Los Angeles Lakers",
        "Cleveland Cavaliers", "Toronto Raptors", "Miami Heat", "Minnesota Timberwolves",
        "Phoenix Suns", "Oklahoma City Thunder"
    ]
    
    def __init__(self, main_url):
        self.main_url = main_url
        self.pdf_url = None
        self.pdf_path = "latest_nba_injury_report.pdf"
        self.csv_path = "nba_injury_report.csv"

    def get_latest_report_url(self):
        """Fetch the latest injury report URL from the NBA website."""
        response = requests.get(self.main_url)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            update_section = soup.find("div", class_="col-xs-12 post-injury")
            if update_section:
                links = update_section.find_all("a")
                if links:
                    self.pdf_url = links[-1]['href']
                    print(f"Latest Report URL: {self.pdf_url}")
                    return True
        print("Failed to fetch the latest injury report URL.")
        return False

    def download_pdf(self):
        """Download the injury report PDF from the fetched URL."""
        if not self.pdf_url:
            print("No PDF URL to download.")
            return False
        response = requests.get(self.pdf_url)
        if response.status_code == 200:
            with open(self.pdf_path, "wb") as file:
                file.write(response.content)
            print(f"Downloaded injury report as '{self.pdf_path}'")
            return True
        print("Failed to download the injury report.")
        return False

    def clean_line(self, line):
        """Clean a line of text by removing headers, footers, and extra whitespace."""
        # Remove page headers and footers
        line = re.sub(r'Injury Report:.*?PM', '', line)
        line = re.sub(r'Page \d+ of \d+', '', line)
        # Remove "NOT YET SUBMITTED"
        line = re.sub(r'NOT YET SUBMITTED.*$', '', line)
        # Clean extra whitespace
        line = ' '.join(line.split())
        return line.strip()

    def extract_team_name(self, line):
        """Extract team name from a line of text."""
        for team in self.nba_teams:
            # Remove spaces for matching
            clean_team = team.replace(' ', '')
            clean_line = line.replace(' ', '')
            if clean_team.lower() in clean_line.lower():
                # Find the starting position of the team name in original line
                start_pos = line.lower().find(team.lower())
                # Return the actual team name and the rest of the line
                return team, line[start_pos + len(team):].strip()
        return None, line

    def parse_pdf_to_csv(self):
        """Parse the downloaded PDF and save the data as a CSV."""
        with pdfplumber.open(self.pdf_path) as pdf:
            with open(self.csv_path, "w", newline="", encoding="utf-8") as csv_file:
                writer = csv.writer(csv_file)
                writer.writerow(["Game Date", "Game Time", "Matchup", "Team", "Player Name", "Current Status", "Reason"])
                
                current_game_info = None
                current_team = None
                current_entry = {
                    'player': None,
                    'status': None,
                    'reason': []
                }

                for page in pdf.pages:
                    text = page.extract_text()
                    lines = text.split('\n')
                    
                    for line in lines:
                        line = self.clean_line(line)
                        if not line or "Game Date Game Time" in line:
                            continue

                        # Match game info
                        game_match = re.match(r'(\d{2}/\d{2}/\d{4})\s+(\d{2}:\d{2}\s*\(ET\))\s+([A-Z]{3}@[A-Z]{3})', line)
                        if game_match:
                            if current_entry['player'] and current_team:
                                self._write_entry(writer, current_game_info, current_team, current_entry)
                            current_game_info = game_match.groups()
                            current_entry = {'player': None, 'status': None, 'reason': []}
                            continue

                        # Try to extract team name
                        team_name, remaining_line = self.extract_team_name(line)
                        if team_name:
                            if current_entry['player'] and current_team:
                                self._write_entry(writer, current_game_info, current_team, current_entry)
                            
                            current_team = team_name
                            current_entry = {'player': None, 'status': None, 'reason': []}
                            
                            # Check for player info in the remaining line
                            player_match = re.match(r'([^,]+,\s*[^,]+)\s+(Out|Questionable|Probable|Available|Doubtful)\s*(.*)', remaining_line)
                            if player_match:
                                player, status, reason = player_match.groups()
                                current_entry = {
                                    'player': player.strip(),
                                    'status': status.strip(),
                                    'reason': [reason.strip()] if reason.strip() else []
                                }
                            continue

                        # Match player info
                        player_match = re.match(r'([^,]+,\s*[^,]+)\s+(Out|Questionable|Probable|Available|Doubtful)\s*(.*)', line)
                        if player_match and current_team:
                            if current_entry['player']:
                                self._write_entry(writer, current_game_info, current_team, current_entry)
                            
                            player, status, reason = player_match.groups()
                            current_entry = {
                                'player': player.strip(),
                                'status': status.strip(),
                                'reason': [reason.strip()] if reason.strip() else []
                            }
                        # Continuation of reason
                        elif current_entry['player']:
                            current_entry['reason'].append(line.strip())

                # Write final entry if exists
                if current_entry['player'] and current_team:
                    self._write_entry(writer, current_game_info, current_team, current_entry)

    def _write_entry(self, writer, game_info, team, entry):
        """Helper method to write a complete entry to CSV."""
        if not game_info or not team or not entry['player']:
            return
            
        # Clean up extra text from player name
        player_name = re.sub(r'\d{2}:\d{2}\s*\(ET\)\s*[A-Z]{3}@[A-Z]{3}.*?(?=[A-Z][a-z]+)', '', entry['player'])
        player_name = player_name.strip()
        
        # Join and clean reason text
        reason = ' '.join(entry['reason']).strip()
        reason = re.sub(r'([A-Z][a-z]+,\s*[A-Z][a-z]+\s+(Out|Questionable|Probable|Available|Doubtful))', '', reason)
        reason = re.sub(r'\s+', ' ', reason).strip()
        
        writer.writerow([
            game_info[0],  # Game Date
            game_info[1],  # Game Time
            game_info[2],  # Matchup
            team,
            player_name,
            entry['status'],
            reason
        ])

    def run(self):
        """Main function to run the scraping and parsing process."""
        if self.get_latest_report_url() and self.download_pdf():
            self.parse_pdf_to_csv()
            print(f"Successfully created CSV file at {self.csv_path}")

# Instantiate and run the scraper
if __name__ == "__main__":
    scraper = NBAInjuryReportScraper(main_url="https://official.nba.com/nba-injury-report-2024-25-season/")
    scraper.run()

In [1]:
import requests
from bs4 import BeautifulSoup
import pdfplumber
import csv
import os
import re

class NBAInjuryReportScraper:
    nba_teams = [
        "Brooklyn Nets", "Philadelphia 76ers", "Boston Celtics", "Washington Wizards",
        "Golden State Warriors", "New Orleans Pelicans", "Atlanta Hawks", "Chicago Bulls",
        "Indiana Pacers", "Milwaukee Bucks", "Portland Trail Blazers", "Houston Rockets",
        "Dallas Mavericks", "Denver Nuggets", "Sacramento Kings", "LA Clippers",
        "New York Knicks", "Utah Jazz", "Detroit Pistons", "Orlando Magic", "Charlotte Hornets",
        "Memphis Grizzlies", "San Antonio Spurs", "Los Angeles Lakers",
        "Cleveland Cavaliers", "Toronto Raptors", "Miami Heat", "Minnesota Timberwolves",
        "Phoenix Suns", "Oklahoma City Thunder"
    ]
    
    def __init__(self, main_url):
        self.main_url = main_url
        self.pdf_url = None
        self.pdf_path = "latest_nba_injury_report.pdf"
        self.csv_path = "nba_injury_report.csv"

    def get_latest_report_url(self):
        """Fetch the latest injury report URL from the NBA website."""
        response = requests.get(self.main_url)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            update_section = soup.find("div", class_="col-xs-12 post-injury")
            if update_section:
                links = update_section.find_all("a")
                if links:
                    self.pdf_url = links[-1]['href']
                    print(f"Latest Report URL: {self.pdf_url}")
                    return True
        print("Failed to fetch the latest injury report URL.")
        return False

    def download_pdf(self):
        """Download the injury report PDF from the fetched URL."""
        if not self.pdf_url:
            print("No PDF URL to download.")
            return False
        response = requests.get(self.pdf_url)
        if response.status_code == 200:
            with open(self.pdf_path, "wb") as file:
                file.write(response.content)
            print(f"Downloaded injury report as '{self.pdf_path}'")
            return True
        print("Failed to download the injury report.")
        return False

    def clean_line(self, line):
        """Clean a line of text."""
        # Remove headers and page numbers
        line = re.sub(r'Page \d+ of \d+', '', line)
        line = re.sub(r'Injury Report:.*?PM', '', line)
        line = re.sub(r'NOT YET SUBMITTED', '', line)
        # Remove extra spaces and trim
        return ' '.join(line.split()).strip()

    def _get_team_from_matchup(self, matchup, first_team=True):
        """Get full team name from matchup code."""
        try:
            team_code = matchup.split('@')[0] if first_team else matchup.split('@')[1]
            for team in self.nba_teams:
                if team.split()[-1][:3].upper() == team_code:
                    return team
            return None
        except:
            return None

    def parse_pdf_to_csv(self):
        """Parse the downloaded PDF and save the data as a CSV."""
        with pdfplumber.open(self.pdf_path) as pdf:
            with open(self.csv_path, "w", newline="", encoding="utf-8") as csv_file:
                writer = csv.writer(csv_file)
                writer.writerow(["Game Date", "Game Time", "Matchup", "Team", "Player Name", "Current Status", "Reason"])
                
                current_date = None
                current_time = None
                current_matchup = None
                current_team = None
                player_data = []

                for page in pdf.pages:
                    text = page.extract_text()
                    lines = text.split('\n')
                    
                    for line in lines:
                        line = self.clean_line(line)
                        if not line or "Game Date" in line:
                            continue

                        # Match game information
                        game_match = re.search(r'(\d{2}/\d{2}/\d{4})\s+(\d{2}:\d{2}\s*\(ET\))\s+([A-Z]{3}@[A-Z]{3})', line)
                        if game_match:
                            current_date = game_match.group(1)
                            current_time = game_match.group(2)
                            current_matchup = game_match.group(3)
                            # Extract team name if present in the same line
                            remaining = line[game_match.end():].strip()
                            if remaining:
                                for team in self.nba_teams:
                                    if team.lower() in remaining.lower():
                                        current_team = team
                                        break
                            continue

                        # Match team name only lines
                        team_found = False
                        for team in self.nba_teams:
                            if team.lower() in line.lower() and len(line) < len(team) + 10:
                                current_team = team
                                team_found = True
                                break
                        if team_found:
                            continue

                        # Match player information
                        player_match = re.search(r'([A-Za-z\s\'\.-]+(?:,\s*[A-Za-z\s\'\.-]+)?)\s+(Out|Questionable|Probable|Available|Doubtful)\s*(.*)', line)
                        if player_match:
                            player_name = player_match.group(1).strip()
                            status = player_match.group(2).strip()
                            reason = player_match.group(3).strip()

                            writer.writerow([
                                current_date,
                                current_time,
                                current_matchup,
                                current_team,
                                player_name,
                                status,
                                reason
                            ])
                            continue

                        # Handle continuation of reason from previous entry
                        if player_data:
                            last_entry = player_data[-1]
                            last_entry['reason'] += ' ' + line.strip()

    def run(self):
        """Main function to run the scraping and parsing process."""
        if self.get_latest_report_url() and self.download_pdf():
            self.parse_pdf_to_csv()
            print(f"Successfully created CSV file at {self.csv_path}")

# Instantiate and run the scraper
if __name__ == "__main__":
    scraper = NBAInjuryReportScraper(main_url="https://official.nba.com/nba-injury-report-2024-25-season/")
    scraper.run()

ConnectionError: HTTPSConnectionPool(host='official.nba.com', port=443): Max retries exceeded with url: /nba-injury-report-2024-25-season/ (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000202238379D0>: Failed to resolve 'official.nba.com' ([Errno 11001] getaddrinfo failed)"))